In [5]:
import requests
import base64
import os
from typing import Dict, List, Optional

# ====================== 配置项（按需修改）======================
IP = "10.120.17.131"  # 服务器IP
PORT = 8000            # 服务端口
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
TIMEOUT = 60           # 超时时间（秒）
ALLOWED_IMAGE_FORMATS = ["jpg", "jpeg", "png", "bmp"]  # 支持的图片格式

# 服务器地址
SERVER_URL = f"http://{IP}:{PORT}/v1/chat/completions"
HEALTH_CHECK_URL = f"http://{IP}:{PORT}/health"

def image_to_base64(image_path: str) -> str:
    """
    将图片转为base64编码（增加格式校验和异常处理）
    """
    # 1. 校验文件是否存在
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片文件不存在：{image_path}")
    
    # 2. 校验图片格式
    file_ext = image_path.split(".")[-1].lower()
    if file_ext not in ALLOWED_IMAGE_FORMATS:
        raise ValueError(
            f"不支持的图片格式：{file_ext}，仅支持 {ALLOWED_IMAGE_FORMATS}"
        )
    
    # 3. 读取并编码
    try:
        with open(image_path, "rb") as f:
            return base64.b64encode(f.read()).decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"图片编码失败：{str(e)}")

def check_server_health() -> bool:
    """
    检查服务器健康状态（调用前先验证服务是否可用）
    """
    try:
        response = requests.get(HEALTH_CHECK_URL, timeout=10)
        if response.status_code == 200:
            health_info = response.json()
            print(f"✅ 服务器健康状态：{health_info}")
            return health_info.get("status") == "healthy" and health_info.get("model_loaded")
        else:
            print(f"❌ 服务器健康检查失败，状态码：{response.status_code}")
            return False
    except Exception as e:
        print(f"❌ 无法连接到服务器：{str(e)}")
        return False

def call_qwen_vl(
    text_prompt: str,
    image_path: Optional[str] = None  # 可选：不传图片则为纯文本调用
) -> str:
    """
    调用Qwen3-VL服务（支持纯文本/图文混合模式）
    """
    # 1. 先检查服务器健康状态
    if not check_server_health():
        return "❌ 服务器未就绪，无法调用"
    
    # 2. 构造请求内容
    content = [{"type": "text", "text": text_prompt}]
    
    # 3. 如有图片则添加base64编码的图片
    if image_path:
        try:
            image_b64 = image_to_base64(image_path)
            content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/{image_path.split('.')[-1].lower()};base64,{image_b64}"}
            })
            print(f"📸 已加载图片：{image_path}")
        except Exception as e:
            return f"❌ 图片处理失败：{str(e)}"
    
    # 4. 构造完整请求体
    payload: Dict = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": content}],
        "max_tokens": 1024,
        "temperature": 0.7,
        "top_p": 0.8,  # 新增：适配Qwen3-VL的采样参数
        "repetition_penalty": 1.05  # 新增：减少重复生成
    }
    
    # 5. 发送请求并处理响应
    try:
        print(f"📡 正在调用服务器：{SERVER_URL}")
        response = requests.post(
            SERVER_URL,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=TIMEOUT
        )
        
        # 打印完整响应（便于调试）
        print(f"🔍 响应状态码：{response.status_code}")
        print(f"🔍 原始响应内容：{response.text}")
        
        response.raise_for_status()  # 抛出HTTP错误
        result = response.json()
        
        # 6. 解析回复
        return result["choices"][0]["message"]["content"]
    
    except requests.exceptions.HTTPError as e:
        return f"❌ HTTP请求失败：{str(e)}\n响应详情：{response.text if 'response' in locals() else '无'}"
    except requests.exceptions.Timeout:
        return f"❌ 请求超时（超过{TIMEOUT}秒）"
    except requests.exceptions.ConnectionError:
        return f"❌ 无法连接到服务器 {IP}:{PORT}，请检查网络或端口是否开放"
    except KeyError as e:
        return f"❌ 响应解析失败（字段缺失）：{str(e)}\n原始响应：{response.text if 'response' in locals() else '无'}"
    except Exception as e:
        return f"❌ 调用异常：{str(e)}"

if __name__ == "__main__":
    # ====================== 测试模式选择（按需注释/取消注释）======================
    # 模式1：图文混合调用（默认）
    test_image_path = "test.png"  # 替换为你的图片路径
    test_prompt = "请详细描述这张图片的内容，包括物体、颜色、场景等"
    
    # 模式2：纯文本调用（调试用，排除图片问题）
    # test_image_path = None
    # test_prompt = "你好，请介绍一下自己的功能和特点"
    
    # ====================== 执行调用并打印结果 ======================
    print("=" * 50)
    print("📝 提问内容：")
    print(test_prompt)
    if test_image_path:
        print(f"🖼️ 图片路径：{test_image_path}")
    print("=" * 50)
    
    # 调用服务
    reply = call_qwen_vl(test_prompt, test_image_path)
    
    print("\n🤖 Qwen3-VL 回复：")
    print(reply)
    print("=" * 50)

📝 提问内容：
请详细描述这张图片的内容，包括物体、颜色、场景等
🖼️ 图片路径：test.png
✅ 服务器健康状态：{'status': 'healthy', 'model_loaded': True, 'gpu_count': 4, 'cuda_available': True, 'model_name': 'Qwen/Qwen3-VL-2B-Instruct'}
📸 已加载图片：test.png
📡 正在调用服务器：http://10.120.17.131:8000/v1/chat/completions
🔍 响应状态码：200
🔍 原始响应内容：{"id":"chat-834671","object":"chat.completion","created":1769361376,"model":"Qwen/Qwen3-VL-2B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"这是一张充满活力与喜悦的宠物照片。画面中心是一只毛茸茸的小白狗，它正从镜头前跑过，充满了动感和快乐的气息。\n\n- **主体**：这只小狗体型娇小，全身覆盖着浓密蓬松的白色长毛，看起来像一只“雪绒花”或“小雪人”。它的耳朵直立，呈三角形，显得非常机灵。眼睛是明亮的深色，瞳孔在阳光下闪着光，眼神中透露出好奇和兴奋。鼻子小巧而黑亮，嘴巴微张，粉色的舌头伸出，仿佛正在开心地叫唤或喘气。它最引人注目的特征是它的一只前爪高高举起，做出一个可爱的“打招呼”的姿势，像是在向镜头示意。\n- **配饰**：小狗脖子上系着一条蓝色的带子，上面有白色的图案，可能是条纹或星星，为它增添了一抹活泼的色彩。\n- **环境**：背景是一个模糊的自然林地，可以看到绿色的树木和灌木丛，以及散落于草地上的枯叶。前景则是清晰可见的鲜绿草叶，与小狗洁白的皮毛形成鲜明对比，使小狗成为整个画面的焦点。\n- **光线与色调**：整张照片沐浴在温暖柔和的日光下，光线从上方照射下来，在小狗身上形成了漂亮的轮廓光。整体色调以白色、绿色为主，辅以蓝色点缀，营造出一种清新、宁静而又充满生机的氛围。\n\n总而言之，这张照片通过捕捉小狗奔跑时的瞬间，生动地传达了宠物的纯真、快乐与活力，给人带来愉悦的感受。\n"},"finish_rea